In [1]:
%load_ext autoreload
%autoreload 2

from typing import Tuple, Optional
import os
import glob
import sys

sys.path.append('./sims')

import numpy as np
import scipy
import matplotlib.pyplot as plt

import initial_stream
import subhalo_orbit
import stream_impact
from rotation_matrix import obs_from_pos6d

In [22]:
def simulate_stream(
    r, phi, vphi, vz, M_sat, tmax, t_a, phi_a, rs_sat, pid,
):
    """ Simulate a stream with a subhalo impact. 

    Parameters
    ----------
    r : float
        Impact parameter in kpc (distance from stream to sat)
    phi : float
        Angle around stream in dec
    vphi : float
        Velocity around stream in km/s
    vz : float
        Velocity along stream in km/s
    M_sat : float
        Mass of subhalo in 1e10 Msun
    tmax : float
        How long stream disrupts in Gyr
    t_a : float
        Time since interaction in Gyr
    phi_a : float
        Interaction point along stream in deg, phi=0 is progenitor location (try -20 to 10)
    rs_sat : float
        Scale radius of subhalo in kpc, can be adjusted along with M_sat using equation 15 in erkal et al. 2015
    pid : int
        Index included in saved filenames

    Returns
    -------
    """
    # TODO: this hack needs to be streamlined better
    DEFAULT_STREAM_PARAMS = dict(
        mu_phi1cosphi2_prog=-0.38297458,
        mu_phi2_prog=-0.87059476,
        rv_prog=-109.48359169,
        dist_prog=21.8659734,
        phi2_prog=0.70106313,
        M_LMC=15,
    )

    # Start simulation
    pid = int(pid)
    SH_x, SH_y, SH_z, SH_vx, SH_vy, SH_vz, _ = initial_stream.chi2_eval(
        tapproach=t_a, tmax=tmax, pid=pid, **DEFAULT_STREAM_PARAMS)
    x_sat, y_sat, z_sat, vx_sat, vy_sat, vz_sat = subhalo_orbit.chi2_eval(
        SH_x, SH_y, SH_z, SH_vx, SH_vy, SH_vz, r, phi, vphi, vz, 
        tmax, t_a, phi_a, pid)
    chi = stream_impact.chi2_eval(
        x_sat=x_sat, y_sat=y_sat, z_sat=z_sat, vx_sat=vx_sat, vy_sat=vy_sat, 
        vz_sat=vz_sat, tmax=tmax, M_sat=M_sat, sr=rs_sat, pid=pid, 
        **DEFAULT_STREAM_PARAMS)

    return 

def run_single_sim(params, pid):

    # check if the necessary directories exist, if not, create them
    os.makedirs('orbits', exist_ok=True)
    os.makedirs('pre_impact', exist_ok=True)
    os.makedirs('final_stream', exist_ok=True)
    os.makedirs('final_coords', exist_ok=True)

    logM_sat, vz = params
    M_sat = 10**logM_sat
    rs_sat = 1.05 * (M_sat * 100)**0.5
    
    # pid = calculate_pid(logM_sat, vz) 
    
    r = 0.2 # impact parameter in kpc (distance from stream to sat)
    phi = 250 # angle around stream in dec
    vphi = 35  # velocity around stream in km/s
    t_a = 0.2 # time since interaction in Gyr
    # interaction point along stream in deg, phi=0 is progenitor location (try -20 to 10)
    phi_a = -4 
    # rs_sat = 0.3 # scale radius of subhalo in kpc, can be adjusted along with M_sat using equation 15 in erkal et al. 2015
    tmax = 4  # how long stream disrupts in Gyr

    simulate_stream(r, phi, vphi, vz, M_sat, tmax, t_a, phi_a, rs_sat, pid)


In [23]:
run_single_sim([0.8, -48], 0)

Setup pid: 0 with tmax=4.000000, t_approach=0.200000 
Sanity check : Initializing potential
tmax: 3.886260
final_t: 3.886260
diff: 0.000000
Running orbit with r=0.2,phi=250,vphi=35,vz=-48,tapproach=0.2,phiapproach=-4,pid=0
Orbit pid: 0 with x=-13.062827, y=-33.631338, z=-23.730192, vx=-49.037487, vy=8.093972, vz=-35.201184, tmax=0.204642,  
Sanity check : Initializing potential
Impact with scale radius:  26.374807530850592
Impact pid: 0 with M_sat=6.309573, rs_sat=26.374808,  
Sanity check : Initializing potential


In [ ]:

def run_sims(nsims=1000):
    pool = Pool()
    logM_array = np.random.uniform(-5,1,nsims)
    vz_array = np.random.uniform(-50,0,nsims)
    pool.map(run_sim, zip(logM_array, vz_array))        



def calculate_pid(logM_sat, vz):
    pid = '%i'%(logM_sat*1000) + '%i'%(-1*vz*1000)
    return int(pid)

def calculate_pid_old(log_Msat, vz):
    pid = 0
    log_Msat_round = round(log_Msat, 3)
    vz_round = round(vz, 3)
    if log_Msat_round >= 0:
        pid = log_Msat_round * 1e6
    else: 
        pid = abs(log_Msat_round) * 1e6 + 1e7
    if  vz_round >= 0: 
        pid += vz_round * 10
    else:
        pid += abs(vz_round) * 10 + 1e3
    return int(pid)

def simulate_stream(r, phi, vphi, vz, M_sat, tmax, t_a, phi_a, rs_sat, pid):
    SH_x, SH_y, SH_z, SH_vx, SH_vy, SH_vz, dunno = initial_stream.chi2_eval(-0.38297458,   -0.87059476, -109.48359169,   21.8659734 , 0.70106313,15,t_a,tmax,int(pid))
    sat_x, sat_y, sat_z, sat_vx, sat_vy, sat_vz = subhalo_orbit.chi2_eval(SH_x, SH_y, SH_z, SH_vx, SH_vy, SH_vz,r,phi,vphi,vz,tmax,t_a,phi_a,int(pid))
    chi = stream_impact.chi2_eval(-0.38297458,   -0.87059476, -109.48359169,   21.8659734 , 0.70106313,15, sat_x, sat_y, sat_z, sat_vx, sat_vy, sat_vz, tmax,M_sat,rs_sat,int(pid))

    os.remove('orbits/orbit_%i.txt' %pid)
    os.remove('pre_impact/pre_impact_%i.txt' %pid)
    os.remove('final_coords/final_coords_%i.txt' %pid)

    # save observables as a new file after simulating the stream
   
    # save_observables_hdf5(pid)
    save_observables_hdf5(pid)


R_phi12_radec = np.array([[0.83697865, 0.29481904, -0.4610298], 
                          [0.51616778, -0.70514011, 0.4861566], 
                          [0.18176238, 0.64487142, 0.74236331]])

def save_observables_txt(pid):
    data = np.genfromtxt(f'final_stream/final_stream_{pid}.txt')
    phi1,phi2,dist,pm1,pm2,vr = obs_from_pos6d(data[:,:3],data[:,3:6],R_phi12_radec)
    observables = np.vstack((phi1, phi2, dist, pm1, pm2, vr))
    np.savetxt(f'final_observables/{pid}.txt', observables)

def save_observables_hdf5(pid):
    data = np.genfromtxt(f'final_stream/final_stream_{pid}.txt')
    phi1,phi2,dist,pm1,pm2,vr = obs_from_pos6d(data[:,:3],data[:,3:6],R_phi12_radec)
    phi1 = phi1.astype('float32')
    phi2 = phi2.astype('float32')
    dist = dist.astype('float32')
    pm1 = pm1.astype('float32')
    pm2 = pm2.astype('float32')
    vr = vr.astype('float32')

    f = h5py.File(f'final_observables/{pid}.hdf5', 'w')
    f.create_dataset('phi1', data = phi1, compression = 'gzip')
    f.create_dataset('phi2', data = phi2, compression = 'gzip')
    f.create_dataset('dist', data = dist, compression = 'gzip')
    f.create_dataset('pm1', data = pm1, compression = 'gzip')
    f.create_dataset('pm2', data = pm2, compression = 'gzip')
    f.create_dataset('vr', data = vr, compression = 'gzip')
    f.close()

def read_observables_txt(txtfile):
    data = np.genfromtxt(txtfile)
    return data[0], data[1], data[2], data[3], data[4], data[5]

def read_observables_hdf5(hdf5file):
    f = h5py.File(hdf5file, 'r')
    phi1 = np.array(f.get('phi1'))
    phi2 = np.array(f.get('phi1'))
    dist = np.array(f.get('dist'))
    pm1 = np.array(f.get('phi1'))
    pm2 = np.array(f.get('phi1'))
    vr = np.array(f.get('vr'))
    f.close()
    return phi1, phi2, dist, pm1, pm2, vr



def calculate_parameters_from_pid(pid):
    # pid as a string taken from filename
    if pid[0]=='-':
        log_Msat = -(int(pid[1:5])/1000)
        vz = -(int(pid[5:])/1000)

    else:
        log_Msat = int(pid[:3])/1000
        vz = -(int(pid[3:])/1000)

    return log_Msat, vz

if __name__ == '__main__':
    run_sims(nsims=10000) 
